# Bloc 8 — SQL & Energy Data Engineering

## Étape 1 — PostgreSQL et architecture de la base

### Partie A — Mise en pratique

## Étape 2 — Ingestion des données énergie dans PostgreSQL

### Partie A — Mise en pratique

In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Program Files\Python312\python.exe
3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]


In [2]:
%pip install "psycopg[binary]"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import joblib
import psycopg
import getpass
from io import StringIO

In [4]:
POSTGRES_PASSWORD = getpass.getpass(
    "Mot de passe PostgreSQL : "
)

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="energy_forecasting",
    user="postgres",
    password=POSTGRES_PASSWORD
)

print("Connexion PostgreSQL réussie.")


Connexion PostgreSQL réussie.


In [5]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            current_database(),
            current_user;
    """)

    print(cur.fetchone())

('energy_forecasting', 'postgres')


In [6]:
import pandas as pd
import numpy as np
import joblib

y = joblib.load(
    "data_processed/y_block3_weather.joblib"
)

print("Nombre total :", len(y))
print("Début :", y.index.min())
print("Fin :", y.index.max())

Nombre total : 227567
Début : 2013-01-07 23:00:00+00:00
Fin : 2025-12-31 22:00:00+00:00


In [7]:
consumption_df = pd.DataFrame({
    "timestamp_utc": y.index,
    "consumption_mw": np.asarray(y)
})

consumption_df["timestamp_utc"] = pd.to_datetime(
    consumption_df["timestamp_utc"],
    utc=True
)

# On protège toujours 2026
consumption_df = consumption_df[
    consumption_df["timestamp_utc"]
    < pd.Timestamp("2026-01-01", tz="UTC")
].copy()

consumption_df = (
    consumption_df
    .dropna()
    .sort_values("timestamp_utc")
    .drop_duplicates(
        subset="timestamp_utc",
        keep="last"
    )
)

print(
    "Observations à envoyer :",
    len(consumption_df)
)

print(
    "Période :",
    consumption_df["timestamp_utc"].min(),
    "→",
    consumption_df["timestamp_utc"].max()
)

Observations à envoyer : 227567
Période : 2013-01-07 23:00:00+00:00 → 2025-12-31 22:00:00+00:00


In [8]:
with conn.cursor() as cur:

    cur.execute("""
        CREATE TEMP TABLE staging_consumption (
            timestamp_utc TIMESTAMPTZ,
            consumption_mw DOUBLE PRECISION
        );
    """)

    with cur.copy("""
        COPY staging_consumption (
            timestamp_utc,
            consumption_mw
        )
        FROM STDIN
    """) as copy:

        for row in consumption_df.itertuples(
            index=False,
            name=None
        ):
            copy.write_row(row)

    cur.execute("""
        INSERT INTO energy.consumption (
            timestamp_utc,
            consumption_mw
        )

        SELECT
            timestamp_utc,
            consumption_mw

        FROM staging_consumption

        ON CONFLICT (timestamp_utc)

        DO UPDATE SET
            consumption_mw =
                EXCLUDED.consumption_mw;
    """)

conn.commit()

print("Import terminé.")

Import terminé.


In [9]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            COUNT(*),
            MIN(timestamp_utc),
            MAX(timestamp_utc),
            AVG(consumption_mw)

        FROM energy.consumption;
    """)

    result = cur.fetchone()


print("Nombre de lignes :", result[0])
print("Premier timestamp :", result[1])
print("Dernier timestamp :", result[2])
print("Consommation moyenne :", result[3])

Nombre de lignes : 227567
Premier timestamp : 2013-01-07 23:00:00+00:00
Dernier timestamp : 2025-12-31 22:00:00+00:00
Consommation moyenne : 52795.8239837059


In [10]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            COUNT(*)
            -
            COUNT(DISTINCT timestamp_utc)

        FROM energy.consumption;
    """)

    duplicates = cur.fetchone()[0]

print("Doublons :", duplicates)

Doublons : 0


In [11]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            DATE_TRUNC(
                'month',
                timestamp_utc
            ) AS month,

            AVG(consumption_mw)
                AS mean_consumption,

            MAX(consumption_mw)
                AS peak_consumption,

            MIN(consumption_mw)
                AS min_consumption

        FROM energy.consumption

        WHERE timestamp_utc >= '2025-01-01'
          AND timestamp_utc <  '2026-01-01'

        GROUP BY month

        ORDER BY month;
    """)

    rows = cur.fetchall()


monthly_sql = pd.DataFrame(
    rows,
    columns=[
        "month",
        "mean_consumption",
        "peak_consumption",
        "min_consumption"
    ]
)

display(monthly_sql)

,month,mean_consumption,peak_consumption,min_consumption
0,2025-01-01 00:00:00+00:00,66550.024194,87716.0,47785.0
1,2025-02-01 00:00:00+00:00,62922.706845,79333.0,44528.0
2,2025-03-01 00:00:00+00:00,54802.617608,70845.0,40339.0
3,2025-04-01 00:00:00+00:00,46216.039583,58901.0,33589.0
4,2025-05-01 00:00:00+00:00,41947.282930,53049.0,30344.0
5,2025-06-01 00:00:00+00:00,43965.343750,60470.0,30306.0
6,2025-07-01 00:00:00+00:00,44235.826613,59754.0,31214.0
7,2025-08-01 00:00:00+00:00,41574.760081,52152.0,30398.0
8,2025-09-01 00:00:00+00:00,43641.819444,54894.0,31035.0
9,2025-10-01 00:00:00+00:00,47018.753360,58557.0,33929.0


## Étape 3 — Météo dans PostgreSQL

In [12]:
X = joblib.load(
    "data_processed/X_block3_weather_nonlinear.joblib"
)

print("Shape :", X.shape)
print("Nombre de colonnes :", len(X.columns))

Shape : (227567, 429)
Nombre de colonnes : 429


In [13]:
weather_keywords = [
    "temp",
    "temperature",
    "humid",
    "humidity",
    "wind",
    "precip",
    "rain",
    "paris",
    "lille",
    "lyon",
    "marseille",
    "bordeaux"
]

weather_columns = [
    col
    for col in X.columns
    if any(
        key in col.lower()
        for key in weather_keywords
    )
]

print(
    "Nombre de variables météo trouvées :",
    len(weather_columns)
)

for col in weather_columns:
    print(col)

Nombre de variables météo trouvées : 11
temp_lag_1h
temp_lag_24h
temp_lag_7d
temp_mean_24h
temp_mean_7d
temp_squared
temp_cubic
temp_hour_sin
temp_hour_cos
temp_month_sin
temp_month_cos


In [14]:
from pathlib import Path

files = sorted(
    Path("data_processed")
    .glob("*")
)

for file in files:
    print(file.name)

block7_residuals_2025.joblib
df_model_block1.joblib
feature_lists.joblib
open-meteo-48.89N2.29E36m.csv
split_config.joblib
weather_cache
weather_france_hourly.joblib
X_block1_final.joblib
X_block3_weather.joblib
X_block3_weather_nonlinear.joblib
y_block1_final.joblib
y_block3_weather.joblib


### Conclusion — Ingestion PostgreSQL de la consommation

La série de consommation a été intégrée avec succès dans PostgreSQL.

La table contient :

$$
227\,567
$$

observations entre janvier 2013 et décembre 2025.

Aucun doublon temporel n'est présent et aucune observation de 2026
n'a été importée.

PostgreSQL est désormais configuré explicitement en UTC, ce qui
garantit la cohérence entre la base de données et les séries utilisées
dans le pipeline Machine Learning.

Les agrégations mensuelles peuvent maintenant être réalisées
directement par SQL.

In [15]:
import joblib
import pandas as pd

weather_raw = joblib.load(
    "data_processed/weather_france_hourly.joblib"
)

print("Type :", type(weather_raw))

if hasattr(weather_raw, "shape"):
    print("Shape :", weather_raw.shape)

if hasattr(weather_raw, "index"):
    print("Index :", type(weather_raw.index))
    print(
        "Période :",
        weather_raw.index.min(),
        "→",
        weather_raw.index.max()
    )

if hasattr(weather_raw, "columns"):
    print("\nColonnes :")
    for col in weather_raw.columns:
        print(col)

print("\nAperçu :")
display(
    weather_raw.head()
)

Type : <class 'pandas.DataFrame'>
Shape : (122736, 6)
Index : <class 'pandas.DatetimeIndex'>
Période : 2012-01-01 00:00:00 → 2025-12-31 23:00:00

Colonnes :
Bordeaux
Lille
Lyon
Marseille
Paris
temp_france

Aperçu :


city,Bordeaux,Lille,Lyon,Marseille,Paris,temp_france
time,,,,,,
2012-01-01 00:00:00,12.7,11.2,8.5,12.0,12.0,11.28
2012-01-01 01:00:00,12.7,11.2,8.5,11.8,12.1,11.26
2012-01-01 02:00:00,12.6,11.1,8.5,11.7,12.1,11.20
2012-01-01 03:00:00,12.5,11.0,8.3,11.5,12.1,11.08
2012-01-01 04:00:00,12.5,11.2,7.9,11.4,12.0,11.00


In [16]:
if isinstance(weather_raw, pd.DataFrame):

    print("\nValeurs manquantes :")

    missing = (
        weather_raw
        .isna()
        .sum()
        .sort_values(
            ascending=False
        )
    )

    print(
        missing[
            missing > 0
        ]
    )


Valeurs manquantes :
Series([], dtype: int64)


In [17]:
print(
    "Timezone :",
    weather_raw.index.tz
)

print(
    "Doublons temporels :",
    weather_raw.index.has_duplicates
)

print(
    "\nPas temporels les plus fréquents :"
)

print(
    weather_raw.index
    .to_series()
    .diff()
    .value_counts()
    .head()
)

Timezone : None
Doublons temporels : False

Pas temporels les plus fréquents :
time
0 days 01:00:00    122735
Name: count, dtype: int64


In [18]:
# ============================================================
# METEO : FORMAT LARGE -> FORMAT LONG
# ============================================================

weather_sql = (
    weather_raw
    .rename(
        columns={
            "temp_france":
                "France"
        }
    )
    .copy()
)

weather_sql.index = pd.to_datetime(
    weather_sql.index
).tz_localize(
    "UTC"
)

weather_sql.index.name = (
    "timestamp_utc"
)

weather_long = (
    weather_sql
    .reset_index()
    .melt(
        id_vars=[
            "timestamp_utc"
        ],
        var_name="city",
        value_name="temperature"
    )
)

# Protection du test final
weather_long = (
    weather_long[
        weather_long[
            "timestamp_utc"
        ]
        <
        pd.Timestamp(
            "2026-01-01",
            tz="UTC"
        )
    ]
    .copy()
)

weather_long = (
    weather_long
    .dropna(
        subset=[
            "timestamp_utc",
            "city",
            "temperature"
        ]
    )
    .sort_values(
        [
            "timestamp_utc",
            "city"
        ]
    )
)

print(
    "Nombre de lignes :",
    len(weather_long)
)

print(
    "Nombre de villes/séries :",
    weather_long["city"].nunique()
)

print(
    weather_long["city"].unique()
)

print(
    "Période :",
    weather_long["timestamp_utc"].min(),
    "→",
    weather_long["timestamp_utc"].max()
)

display(
    weather_long.head(12)
)

Nombre de lignes : 736416
Nombre de villes/séries : 6
<ArrowStringArray>
['Bordeaux', 'France', 'Lille', 'Lyon', 'Marseille', 'Paris']
Length: 6, dtype: str
Période : 2012-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00


,timestamp_utc,city,temperature
0,2012-01-01 00:00:00+00:00,Bordeaux,12.70
613680,2012-01-01 00:00:00+00:00,France,11.28
122736,2012-01-01 00:00:00+00:00,Lille,11.20
245472,2012-01-01 00:00:00+00:00,Lyon,8.50
368208,2012-01-01 00:00:00+00:00,Marseille,12.00
490944,2012-01-01 00:00:00+00:00,Paris,12.00
1,2012-01-01 01:00:00+00:00,Bordeaux,12.70
613681,2012-01-01 01:00:00+00:00,France,11.26
122737,2012-01-01 01:00:00+00:00,Lille,11.20
245473,2012-01-01 01:00:00+00:00,Lyon,8.50


In [19]:
import time

start = time.time()

with conn.cursor() as cur:

    # ==========================================
    # Staging
    # ==========================================

    cur.execute("""
        CREATE TEMP TABLE staging_weather (
            timestamp_utc TIMESTAMPTZ,
            city VARCHAR(50),
            temperature DOUBLE PRECISION
        );
    """)

    # ==========================================
    # COPY
    # ==========================================

    with cur.copy("""
        COPY staging_weather (
            timestamp_utc,
            city,
            temperature
        )
        FROM STDIN
    """) as copy:

        for row in weather_long[
            [
                "timestamp_utc",
                "city",
                "temperature"
            ]
        ].itertuples(
            index=False,
            name=None
        ):

            copy.write_row(row)

    # ==========================================
    # TABLE FINALE
    # ==========================================

    cur.execute("""
        INSERT INTO energy.weather (
            timestamp_utc,
            city,
            temperature
        )

        SELECT
            timestamp_utc,
            city,
            temperature

        FROM staging_weather

        ON CONFLICT (
            timestamp_utc,
            city
        )

        DO UPDATE SET

            temperature =
                EXCLUDED.temperature;
    """)

conn.commit()

elapsed = (
    time.time()
    - start
)

print(
    f"Import météo terminé en {elapsed:.2f} s."
)

Import météo terminé en 16.08 s.


In [20]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            COUNT(*),
            COUNT(DISTINCT city),
            MIN(timestamp_utc),
            MAX(timestamp_utc)

        FROM energy.weather;
    """)

    result = cur.fetchone()


print(
    "Nombre de lignes :",
    result[0]
)

print(
    "Nombre de séries :",
    result[1]
)

print(
    "Premier timestamp :",
    result[2]
)

print(
    "Dernier timestamp :",
    result[3]
)

Nombre de lignes : 736416
Nombre de séries : 6
Premier timestamp : 2012-01-01 00:00:00+00:00
Dernier timestamp : 2025-12-31 23:00:00+00:00


In [21]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            city,
            COUNT(*) AS n,
            AVG(temperature) AS mean_temp,
            MIN(temperature) AS min_temp,
            MAX(temperature) AS max_temp

        FROM energy.weather

        GROUP BY city

        ORDER BY city;
    """)

    rows = cur.fetchall()


weather_sql_summary = pd.DataFrame(
    rows,
    columns=[
        "city",
        "n",
        "mean_temp",
        "min_temp",
        "max_temp"
    ]
)

display(
    weather_sql_summary
)

,city,n,mean_temp,min_temp,max_temp
0,Bordeaux,122736,14.225121,-7.6,41.70
1,France,122736,13.310777,-6.7,36.34
2,Lille,122736,11.386272,-10.3,38.10
3,Lyon,122736,12.829673,-9.9,40.30
4,Marseille,122736,15.880016,-3.7,36.90
5,Paris,122736,12.232800,-9.7,40.70


In [22]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            c.timestamp_utc,
            c.consumption_mw,
            w.temperature

        FROM energy.consumption AS c

        LEFT JOIN energy.weather AS w

            ON c.timestamp_utc =
               w.timestamp_utc

           AND w.city = 'France'

        WHERE
            c.timestamp_utc >=
                '2025-01-01'

        AND c.timestamp_utc <
                '2025-01-03'

        ORDER BY
            c.timestamp_utc;
    """)

    rows = cur.fetchall()


join_example = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "consumption_mw",
        "temperature"
    ]
)

display(
    join_example.head(20)
)

,timestamp_utc,consumption_mw,temperature
0,2025-01-01 00:00:00+00:00,62487.0,3.28
1,2025-01-01 00:30:00+00:00,62236.0,NaN
2,2025-01-01 01:00:00+00:00,61797.0,3.12
3,2025-01-01 01:30:00+00:00,61762.0,NaN
4,2025-01-01 02:00:00+00:00,60405.0,2.88
5,2025-01-01 02:30:00+00:00,59242.0,NaN
6,2025-01-01 03:00:00+00:00,57994.0,3.26
7,2025-01-01 03:30:00+00:00,57219.0,NaN
8,2025-01-01 04:00:00+00:00,56492.0,3.30
9,2025-01-01 04:30:00+00:00,56682.0,NaN


In [23]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            c.timestamp_utc,
            c.consumption_mw,
            w.temperature

        FROM energy.consumption AS c

        LEFT JOIN LATERAL (

            SELECT
                temperature

            FROM energy.weather AS w

            WHERE
                w.city = 'France'

            AND
                w.timestamp_utc
                <=
                c.timestamp_utc

            ORDER BY
                w.timestamp_utc DESC

            LIMIT 1

        ) AS w
        ON TRUE

        WHERE
            c.timestamp_utc >=
                '2025-01-01'

        AND c.timestamp_utc <
                '2025-01-02'

        ORDER BY
            c.timestamp_utc;
    """)

    rows = cur.fetchall()


join_ffill = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "consumption_mw",
        "temperature"
    ]
)

display(
    join_ffill.head(20)
)

,timestamp_utc,consumption_mw,temperature
0,2025-01-01 00:00:00+00:00,62487.0,3.28
1,2025-01-01 00:30:00+00:00,62236.0,3.28
2,2025-01-01 01:00:00+00:00,61797.0,3.12
3,2025-01-01 01:30:00+00:00,61762.0,3.12
4,2025-01-01 02:00:00+00:00,60405.0,2.88
5,2025-01-01 02:30:00+00:00,59242.0,2.88
6,2025-01-01 03:00:00+00:00,57994.0,3.26
7,2025-01-01 03:30:00+00:00,57219.0,3.26
8,2025-01-01 04:00:00+00:00,56492.0,3.30
9,2025-01-01 04:30:00+00:00,56682.0,3.30


In [24]:
print(
    "Températures manquantes :",
    join_ffill[
        "temperature"
    ].isna().sum()
)

Températures manquantes : 0


### Conclusion — Intégration SQL des données météorologiques

Les températures horaires de cinq villes françaises ainsi qu'une
température moyenne nationale ont été intégrées dans PostgreSQL.

La base météo contient six séries :

- Bordeaux ;
- Lille ;
- Lyon ;
- Marseille ;
- Paris ;
- France.

La température France correspond à la moyenne des cinq stations.

Les données météorologiques sont horaires tandis que la consommation
est disponible toutes les 30 minutes.

Une jointure temporelle `LATERAL JOIN` permet de sélectionner la
dernière observation météorologique connue pour chaque instant de
consommation, reproduisant ainsi le principe de forward fill utilisé
dans le pipeline Python.

La base PostgreSQL contient désormais les deux sources principales
du système :

$$
\text{consommation}
+
\text{météo}.
$$


## Étape 4 — Feature Engineering temporel en SQL

### Partie A — Mise en pratique

In [25]:
import time

start = time.time()

with conn.cursor() as cur:

    cur.execute("""
        SET statement_timeout = '5min';

        CREATE INDEX IF NOT EXISTS
            idx_weather_city_timestamp
        ON energy.weather (
            city,
            timestamp_utc DESC
        );

        ANALYZE energy.weather;
    """)

conn.commit()

print(
    f"Index créé en {time.time() - start:.2f} s."
)

Index créé en 1.64 s.


In [26]:
start = time.time()

with conn.cursor() as cur:

    cur.execute("""
        SET statement_timeout = '5min';

        DROP MATERIALIZED VIEW
        IF EXISTS energy.ml_features;

        CREATE MATERIALIZED VIEW
        energy.ml_features
        AS

        WITH base AS (

            SELECT
                c.timestamp_utc,
                c.consumption_mw,
                w.temperature

            FROM energy.consumption AS c

            LEFT JOIN LATERAL (

                SELECT
                    temperature

                FROM energy.weather AS w

                WHERE
                    w.city = 'France'

                    AND
                    w.timestamp_utc
                    <= c.timestamp_utc

                ORDER BY
                    w.timestamp_utc DESC

                LIMIT 1

            ) AS w
            ON TRUE

            WHERE
                c.timestamp_utc
                < '2026-01-01'
        ),


        temporal AS (

            SELECT
                *,

                -- =========================================
                -- LAGS CONSOMMATION
                -- =========================================

                LAG(
                    consumption_mw,
                    1
                ) OVER w
                    AS consumption_lag_30m,

                LAG(
                    consumption_mw,
                    2
                ) OVER w
                    AS consumption_lag_1h,

                LAG(
                    consumption_mw,
                    48
                ) OVER w
                    AS consumption_lag_24h,

                LAG(
                    consumption_mw,
                    336
                ) OVER w
                    AS consumption_lag_7d,


                -- =========================================
                -- MOYENNES PASSEES
                -- On exclut toujours la valeur actuelle
                -- =========================================

                AVG(
                    consumption_mw
                ) OVER (
                    ORDER BY timestamp_utc
                    ROWS BETWEEN
                        6 PRECEDING
                        AND 1 PRECEDING
                )
                    AS consumption_mean_3h,

                AVG(
                    consumption_mw
                ) OVER (
                    ORDER BY timestamp_utc
                    ROWS BETWEEN
                        48 PRECEDING
                        AND 1 PRECEDING
                )
                    AS consumption_mean_24h,

                AVG(
                    consumption_mw
                ) OVER (
                    ORDER BY timestamp_utc
                    ROWS BETWEEN
                        336 PRECEDING
                        AND 1 PRECEDING
                )
                    AS consumption_mean_7d,


                STDDEV_SAMP(
                    consumption_mw
                ) OVER (
                    ORDER BY timestamp_utc
                    ROWS BETWEEN
                        48 PRECEDING
                        AND 1 PRECEDING
                )
                    AS consumption_std_24h,


                -- =========================================
                -- METEO
                -- =========================================

                LAG(
                    temperature,
                    2
                ) OVER w
                    AS temp_lag_1h,

                LAG(
                    temperature,
                    48
                ) OVER w
                    AS temp_lag_24h,

                LAG(
                    temperature,
                    336
                ) OVER w
                    AS temp_lag_7d,

                AVG(
                    temperature
                ) OVER (
                    ORDER BY timestamp_utc
                    ROWS BETWEEN
                        48 PRECEDING
                        AND 1 PRECEDING
                )
                    AS temp_mean_24h

            FROM base

            WINDOW w AS (
                ORDER BY timestamp_utc
            )
        ),


        calendar AS (

            SELECT
                *,

                timestamp_utc
                    AT TIME ZONE
                    'Europe/Paris'
                    AS local_timestamp

            FROM temporal
        )


        SELECT
            *,

            -- =============================================
            -- CALENDRIER LOCAL FRANÇAIS
            -- =============================================

            EXTRACT(
                HOUR
                FROM local_timestamp
            )
                +
            EXTRACT(
                MINUTE
                FROM local_timestamp
            ) / 60.0
                AS local_hour,

            EXTRACT(
                ISODOW
                FROM local_timestamp
            )
                AS day_of_week,

            EXTRACT(
                MONTH
                FROM local_timestamp
            )
                AS month,

            CASE
                WHEN EXTRACT(
                    ISODOW
                    FROM local_timestamp
                ) IN (6, 7)
                THEN 1
                ELSE 0
            END
                AS is_weekend,


            -- =============================================
            -- ENCODAGES CYCLIQUES
            -- =============================================

            SIN(
                2 * PI()
                *
                (
                    EXTRACT(
                        HOUR
                        FROM local_timestamp
                    )
                    +
                    EXTRACT(
                        MINUTE
                        FROM local_timestamp
                    ) / 60.0
                )
                / 24
            )
                AS hour_sin,

            COS(
                2 * PI()
                *
                (
                    EXTRACT(
                        HOUR
                        FROM local_timestamp
                    )
                    +
                    EXTRACT(
                        MINUTE
                        FROM local_timestamp
                    ) / 60.0
                )
                / 24
            )
                AS hour_cos,

            SIN(
                2 * PI()
                *
                EXTRACT(
                    MONTH
                    FROM local_timestamp
                )
                / 12
            )
                AS month_sin,

            COS(
                2 * PI()
                *
                EXTRACT(
                    MONTH
                    FROM local_timestamp
                )
                / 12
            )
                AS month_cos

        FROM calendar

        ORDER BY timestamp_utc;
    """)

conn.commit()

print(
    f"Materialized view créée en "
    f"{time.time() - start:.2f} s."
)

Materialized view créée en 12.67 s.


In [27]:
with conn.cursor() as cur:

    cur.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS
            idx_ml_features_timestamp
        ON energy.ml_features (
            timestamp_utc
        );

        ANALYZE energy.ml_features;
    """)

conn.commit()

print("Index ML créé.")

Index ML créé.


In [28]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            COUNT(*),
            MIN(timestamp_utc),
            MAX(timestamp_utc)

        FROM energy.ml_features;
    """)

    result = cur.fetchone()


print(
    "Nombre de lignes :",
    result[0]
)

print(
    "Début :",
    result[1]
)

print(
    "Fin :",
    result[2]
)

Nombre de lignes : 227567
Début : 2013-01-07 23:00:00+00:00
Fin : 2025-12-31 22:00:00+00:00


In [29]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            timestamp_utc,
            consumption_mw,

            consumption_lag_30m,
            consumption_lag_1h,
            consumption_lag_24h,
            consumption_lag_7d,

            consumption_mean_24h,

            temperature,
            temp_lag_1h,
            temp_lag_24h,

            local_hour,
            day_of_week,
            is_weekend

        FROM energy.ml_features

        WHERE
            timestamp_utc >=
                '2025-01-01'

        LIMIT 10;
    """)

    rows = cur.fetchall()


columns = [
    "timestamp",
    "consumption",
    "lag_30m",
    "lag_1h",
    "lag_24h",
    "lag_7d",
    "mean_24h",
    "temperature",
    "temp_lag_1h",
    "temp_lag_24h",
    "local_hour",
    "dow",
    "weekend"
]

sql_features_preview = pd.DataFrame(
    rows,
    columns=columns
)

display(
    sql_features_preview
)

,timestamp,consumption,lag_30m,lag_1h,lag_24h,lag_7d,mean_24h,temperature,temp_lag_1h,temp_lag_24h,local_hour,dow,weekend
0,2025-01-01 00:00:00+00:00,62487.0,64518.0,64756.0,63310.0,53747.0,66013.020833,3.28,3.28,2.56,1.00000000000000000000,3,0
1,2025-01-01 00:30:00+00:00,62236.0,62487.0,64518.0,62656.0,53326.0,65995.875000,3.28,3.28,2.56,1.50000000000000000000,3,0
2,2025-01-01 01:00:00+00:00,61797.0,62236.0,62487.0,62060.0,52786.0,65987.125000,3.12,3.28,2.44,2.00000000000000000000,3,0
3,2025-01-01 01:30:00+00:00,61762.0,61797.0,62236.0,61793.0,52408.0,65981.645833,3.12,3.28,2.44,2.50000000000000000000,3,0
4,2025-01-01 02:00:00+00:00,60405.0,61762.0,61797.0,60470.0,50774.0,65981.000000,2.88,3.12,2.36,3.00000000000000000000,3,0
5,2025-01-01 02:30:00+00:00,59242.0,60405.0,61762.0,59571.0,49412.0,65979.645833,2.88,3.12,2.36,3.50000000000000000000,3,0
6,2025-01-01 03:00:00+00:00,57994.0,59242.0,60405.0,58612.0,47938.0,65972.791667,3.26,2.88,1.70,4.00000000000000000000,3,0
7,2025-01-01 03:30:00+00:00,57219.0,57994.0,59242.0,58301.0,47070.0,65959.916667,3.26,2.88,1.70,4.50000000000000000000,3,0
8,2025-01-01 04:00:00+00:00,56492.0,57219.0,57994.0,58094.0,46407.0,65937.375000,3.30,3.26,1.54,5.00000000000000000000,3,0
9,2025-01-01 04:30:00+00:00,56682.0,56492.0,57219.0,59264.0,46764.0,65904.000000,3.30,3.26,1.54,5.50000000000000000000,3,0


In [30]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            timestamp_utc,
            consumption_mw,
            consumption_lag_30m,
            consumption_lag_1h,
            consumption_lag_24h

        FROM energy.ml_features

        WHERE
            timestamp_utc =
            '2025-01-02 12:00:00+00';
    """)

    print(
        cur.fetchone()
    )

(datetime.datetime(2025, 1, 2, 12, 0, tzinfo=datetime.timezone.utc), 69987.0, 69224.0, 68292.0, 63194.0)


In [31]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            timestamp_utc,
            consumption_mw

        FROM energy.consumption

        WHERE timestamp_utc IN (
            '2025-01-02 11:30:00+00',
            '2025-01-02 11:00:00+00',
            '2025-01-01 12:00:00+00'
        )

        ORDER BY timestamp_utc;
    """)

    print(
        cur.fetchall()
    )

[(datetime.datetime(2025, 1, 1, 12, 0, tzinfo=datetime.timezone.utc), 63194.0), (datetime.datetime(2025, 1, 2, 11, 0, tzinfo=datetime.timezone.utc), 68292.0), (datetime.datetime(2025, 1, 2, 11, 30, tzinfo=datetime.timezone.utc), 69224.0)]


In [32]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT

            COUNT(*) FILTER (
                WHERE consumption_lag_30m IS NULL
            ),

            COUNT(*) FILTER (
                WHERE consumption_lag_1h IS NULL
            ),

            COUNT(*) FILTER (
                WHERE consumption_lag_24h IS NULL
            ),

            COUNT(*) FILTER (
                WHERE consumption_lag_7d IS NULL
            )

        FROM energy.ml_features;
    """)

    null_counts = cur.fetchone()


print(
    "NULL lag 30 min :",
    null_counts[0]
)

print(
    "NULL lag 1h :",
    null_counts[1]
)

print(
    "NULL lag 24h :",
    null_counts[2]
)

print(
    "NULL lag 7j :",
    null_counts[3]
)

NULL lag 30 min : 1
NULL lag 1h : 2
NULL lag 24h : 48
NULL lag 7j : 336


### Partie C — Analyse et améliorations guidées par la théorie

PostgreSQL est désormais utilisé comme moteur de Feature Engineering
temporel et non plus uniquement comme système de stockage.

Les fonctions fenêtre permettent de construire directement des
variables retardées :

$$
y_{t-1},\quad
y_{t-2},\quad
y_{t-48},\quad
y_{t-336}.
$$

Pour une fréquence de 30 minutes, elles correspondent respectivement
à :

$$
30\text{ min},\quad
1\text{ h},\quad
24\text{ h},\quad
7\text{ jours}.
$$

Les statistiques mobiles sont calculées uniquement sur les
observations antérieures à l'instant prédit afin d'éviter toute fuite
de données.

La météo horaire est alignée sur la fréquence de 30 minutes par une
jointure temporelle utilisant la dernière observation disponible.

Les variables calendaires sont calculées en heure locale française,
tandis que les timestamps restent stockés en UTC.

Enfin, les résultats du Feature Engineering sont stockés dans une
Materialized View afin d'éviter de recalculer systématiquement les
jointures et fonctions fenêtre.

## Étape 5 — SQL Optimization & Production Views

### Partie A — Mise en pratique

In [33]:
with conn.cursor() as cur:

    cur.execute("""
        EXPLAIN ANALYZE

        SELECT
            timestamp_utc,
            consumption_mw,
            consumption_lag_24h,
            consumption_lag_7d,
            temperature

        FROM energy.ml_features

        WHERE
            timestamp_utc >= '2025-01-01'
            AND
            timestamp_utc < '2025-02-01'

        ORDER BY timestamp_utc;
    """)

    plan_index = cur.fetchall()


for row in plan_index:
    print(row[0])

Index Scan using idx_ml_features_timestamp on ml_features  (cost=0.42..85.74 rows=1416 width=40) (actual time=0.028..0.597 rows=1488 loops=1)
  Index Cond: ((timestamp_utc >= '2025-01-01 00:00:00+00'::timestamp with time zone) AND (timestamp_utc < '2025-02-01 00:00:00+00'::timestamp with time zone))
Planning Time: 0.150 ms
Execution Time: 0.677 ms


In [34]:
with conn.cursor() as cur:

    cur.execute("""
        EXPLAIN ANALYZE

        SELECT
            AVG(consumption_mw)

        FROM energy.ml_features;
    """)

    plan_full = cur.fetchall()


for row in plan_full:
    print(row[0])

Finalize Aggregate  (cost=8201.46..8201.47 rows=1 width=8) (actual time=125.080..136.634 rows=1 loops=1)
  ->  Gather  (cost=8201.25..8201.46 rows=2 width=32) (actual time=124.288..136.582 rows=3 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        ->  Partial Aggregate  (cost=7201.25..7201.26 rows=1 width=32) (actual time=55.456..55.459 rows=1 loops=3)
              ->  Parallel Seq Scan on ml_features  (cost=0.00..6964.20 rows=94820 width=8) (actual time=0.712..39.003 rows=75856 loops=3)
Planning Time: 0.119 ms
Execution Time: 136.696 ms


In [35]:
import time
import numpy as np

sql_times = []

for _ in range(5):

    start = time.perf_counter()

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                timestamp_utc,
                consumption_mw,
                consumption_lag_30m,
                consumption_lag_1h,
                consumption_lag_24h,
                consumption_lag_7d,
                consumption_mean_24h,
                temperature,
                temp_lag_1h,
                temp_lag_24h,
                local_hour,
                day_of_week,
                is_weekend

            FROM energy.ml_features

            WHERE
                timestamp_utc >= '2025-01-01'
                AND
                timestamp_utc < '2026-01-01'

            ORDER BY timestamp_utc;
        """)

        rows = cur.fetchall()

    sql_times.append(
        time.perf_counter() - start
    )


print(
    "Nombre de lignes :",
    len(rows)
)

print(
    "Temps SQL :",
    np.round(sql_times, 4)
)

print(
    "Temps SQL médian :",
    np.median(sql_times),
    "s"
)

Nombre de lignes : 17517
Temps SQL : [0.0773 0.0764 0.0813 0.0822 0.0915]
Temps SQL médian : 0.08125139999901876 s


In [36]:
pandas_times = []

for _ in range(5):

    start = time.perf_counter()

    pandas_subset = X.loc[
        (X.index >= "2025-01-01")
        &
        (X.index < "2026-01-01")
    ]

    pandas_times.append(
        time.perf_counter() - start
    )


print(
    "Nombre de lignes pandas :",
    len(pandas_subset)
)

print(
    "Temps pandas :",
    np.round(
        pandas_times,
        4
    )
)

print(
    "Temps pandas médian :",
    np.median(
        pandas_times
    ),
    "s"
)

Nombre de lignes pandas : 17517
Temps pandas : [0.0553 0.0647 0.059  0.0795 0.0567]
Temps pandas médian : 0.05900549999932991 s


In [37]:
with conn.cursor() as cur:

    cur.execute("""
        CREATE OR REPLACE VIEW
            energy.ml_dataset
        AS

        SELECT
            timestamp_utc,

            consumption_mw,

            consumption_lag_30m,
            consumption_lag_1h,
            consumption_lag_24h,
            consumption_lag_7d,

            consumption_mean_3h,
            consumption_mean_24h,
            consumption_mean_7d,
            consumption_std_24h,

            temperature,

            temp_lag_1h,
            temp_lag_24h,
            temp_lag_7d,
            temp_mean_24h,

            local_hour,
            day_of_week,
            month,
            is_weekend,

            hour_sin,
            hour_cos,
            month_sin,
            month_cos

        FROM energy.ml_features

        WHERE
            consumption_lag_7d IS NOT NULL

            AND timestamp_utc
                < '2026-01-01';
    """)

conn.commit()

print(
    "energy.ml_dataset créée."
)

energy.ml_dataset créée.


In [38]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            COUNT(*),
            MIN(timestamp_utc),
            MAX(timestamp_utc)

        FROM energy.ml_dataset;
    """)

    production_summary = (
        cur.fetchone()
    )


print(
    "Nombre de lignes :",
    production_summary[0]
)

print(
    "Début :",
    production_summary[1]
)

print(
    "Fin :",
    production_summary[2]
)

Nombre de lignes : 227231
Début : 2013-01-14 23:00:00+00:00
Fin : 2025-12-31 22:00:00+00:00


In [39]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            COUNT(*) FILTER (
                WHERE consumption_lag_30m IS NULL
            ),

            COUNT(*) FILTER (
                WHERE consumption_lag_1h IS NULL
            ),

            COUNT(*) FILTER (
                WHERE consumption_lag_24h IS NULL
            ),

            COUNT(*) FILTER (
                WHERE consumption_lag_7d IS NULL
            ),

            COUNT(*) FILTER (
                WHERE temperature IS NULL
            )

        FROM energy.ml_dataset;
    """)

    null_production = (
        cur.fetchone()
    )


print(
    "NULL lag 30m :",
    null_production[0]
)

print(
    "NULL lag 1h :",
    null_production[1]
)

print(
    "NULL lag 24h :",
    null_production[2]
)

print(
    "NULL lag 7d :",
    null_production[3]
)

print(
    "NULL température :",
    null_production[4]
)

NULL lag 30m : 0
NULL lag 1h : 0
NULL lag 24h : 0
NULL lag 7d : 0
NULL température : 0


In [40]:
with conn.cursor() as cur:

    cur.execute("""
        SELECT
            timestamp_utc,
            consumption_mw,
            temperature,
            local_hour,
            day_of_week

        FROM energy.ml_dataset

        WHERE
            timestamp_utc >= '2025-01-01'
            AND
            timestamp_utc < '2026-01-01'

        ORDER BY
            consumption_mw DESC

        LIMIT 10;
    """)

    rows = cur.fetchall()


top_peaks = pd.DataFrame(
    rows,
    columns=[
        "timestamp_utc",
        "consumption_mw",
        "temperature",
        "local_hour",
        "day_of_week"
    ]
)

display(
    top_peaks
)

,timestamp_utc,consumption_mw,temperature,local_hour,day_of_week
0,2025-01-14 08:00:00+00:00,87716.0,-3.08,9.00000000000000000000,2
1,2025-01-14 08:30:00+00:00,87689.0,-3.08,9.50000000000000000000,2
2,2025-01-14 07:30:00+00:00,87163.0,-2.98,8.50000000000000000000,2
3,2025-01-14 09:00:00+00:00,87040.0,-3.38,10.00000000000000000000,2
4,2025-01-14 09:30:00+00:00,85724.0,-3.38,10.50000000000000000000,2
5,2025-01-14 07:00:00+00:00,85353.0,-2.98,8.00000000000000000000,2
6,2025-01-14 10:00:00+00:00,84542.0,-2.90,11.00000000000000000000,2
7,2025-01-15 08:30:00+00:00,84391.0,-0.88,9.50000000000000000000,3
8,2025-01-14 18:30:00+00:00,84343.0,1.94,19.50000000000000000000,2
9,2025-01-15 09:00:00+00:00,84319.0,-1.00,10.00000000000000000000,3


### Partie C — Analyse et améliorations guidées par la théorie

L'optimisation SQL ne consiste pas uniquement à ajouter des index.

PostgreSQL choisit automatiquement un plan d'exécution en fonction
du coût estimé de la requête.

Pour une sélection temporelle restreinte, un index sur :

$$
timestamp\_utc
$$

permet de localiser rapidement les observations recherchées.

En revanche, lorsqu'une requête nécessite la lecture d'une grande
partie de la table, un `Sequential Scan` peut être plus efficace
qu'un parcours par index.

La vue `energy.ml_dataset` fournit une interface stable entre la
couche Data Engineering et la couche Machine Learning.

L'architecture devient :

$$
\text{Raw Data}
\rightarrow
\text{PostgreSQL}
\rightarrow
\text{Feature Engineering}
\rightarrow
\text{Materialized View}
\rightarrow
\text{ML Dataset}.
$$

Cette séparation permet aux modèles de ne pas dépendre directement
de la logique d'ingestion et de transformation des données.

### Conclusion — SQL Optimization & Production Views

L'analyse du plan d'exécution confirme que PostgreSQL exploite
correctement les index temporels.

Pour une requête limitée à janvier 2025, PostgreSQL utilise un
`Index Scan` et obtient un temps d'exécution d'environ :

$$
0.68\text{ ms}.
$$

Pour une agrégation sur l'ensemble de la table, PostgreSQL choisit
au contraire un `Parallel Sequential Scan`, ce qui est cohérent
puisque la majorité des observations doit être parcourue.

L'extraction de l'année 2025 depuis PostgreSQL prend environ :

$$
0.081\text{ s}
$$

contre :

$$
0.059\text{ s}
$$

avec pandas lorsque le DataFrame est déjà présent en mémoire.

Cette comparaison ne doit pas être interprétée comme un benchmark
direct : SQL fournit également le stockage persistant, les index,
la reproductibilité des requêtes et une interface adaptée à un
système de production.

La vue `energy.ml_dataset` contient :

$$
227\,231
$$

observations et ne présente aucune valeur manquante sur les variables
temporelles critiques.

La couche Data Engineering est donc prête à être consommée directement
par les futurs composants Machine Learning et API.

In [42]:
import os
import getpass

os.environ["POSTGRES_PASSWORD"] = getpass.getpass(
    "Mot de passe PostgreSQL : "
)

print("Mot de passe PostgreSQL chargé pour cette session.")

Mot de passe PostgreSQL chargé pour cette session.


In [43]:
from src.database.postgres import (
    load_ml_dataset
)

sql_2025 = load_ml_dataset(
    "2025-01-01",
    "2026-01-01"
)

print(
    sql_2025.shape
)

display(
    sql_2025.head()
)

(17517, 23)


,timestamp_utc,consumption_mw,consumption_lag_30m,consumption_lag_1h,consumption_lag_24h,consumption_lag_7d,consumption_mean_3h,consumption_mean_24h,consumption_mean_7d,consumption_std_24h,...,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos
0,2025-01-01 00:00:00+00:00,62487.0,64518.0,64756.0,63310.0,53747.0,64458.166667,66013.020833,61187.241071,4119.255750,...,7.26,3.156667,1.00000000000000000000,3,1,0,0.258819,0.965926,0.5,0.866025
1,2025-01-01 00:30:00+00:00,62236.0,62487.0,64518.0,62656.0,53326.0,64362.166667,65995.875000,61213.252976,4132.437803,...,7.26,3.171667,1.50000000000000000000,3,1,0,0.382683,0.923880,0.5,0.866025
2,2025-01-01 01:00:00+00:00,61797.0,62236.0,62487.0,62060.0,52786.0,64098.833333,65987.125000,61239.770833,4140.097653,...,7.20,3.186667,2.00000000000000000000,3,1,0,0.500000,0.866025,0.5,0.866025
3,2025-01-01 01:30:00+00:00,61762.0,61797.0,62236.0,61793.0,52408.0,63492.166667,65981.645833,61266.589286,4145.575953,...,7.20,3.200833,2.50000000000000000000,3,1,0,0.608761,0.793353,0.5,0.866025
4,2025-01-01 02:00:00+00:00,60405.0,61762.0,61797.0,60470.0,50774.0,62926.000000,65981.000000,61294.428571,4146.244740,...,6.86,3.215000,3.00000000000000000000,3,1,0,0.707107,0.707107,0.5,0.866025


In [44]:
import joblib
import numpy as np
import pandas as pd

block7_state = joblib.load(
    "models/block7_final_state.joblib"
)

print("Modèle :", block7_state["model_name"])
print("\nMétriques :")
print(block7_state["metrics_2025"])

Modèle : Ridge + MLP1 + Patch Transformer + Refined Event Expert

Métriques :
{'MSE': 31308.66995957857, 'RMSE': np.float64(176.94256118746154), 'MAE': 127.8516195648178, 'MAPE_%': np.float64(0.25761173413837657), 'R2': 0.9997223768098071}


In [45]:
metrics = block7_state["metrics_2025"]

def get_metric(dictionary, *names):
    for name in names:
        if name in dictionary:
            return float(dictionary[name])
    raise KeyError(
        f"Métrique introuvable parmi : {names}"
    )

MODEL_NAME = block7_state["model_name"]

mse = get_metric(metrics, "MSE")
rmse = get_metric(metrics, "RMSE")
mae = get_metric(metrics, "MAE")
mape = get_metric(metrics, "MAPE", "MAPE_%")
r2 = get_metric(metrics, "R2", "R²")

print("MSE  :", mse)
print("RMSE :", rmse)
print("MAE  :", mae)
print("MAPE :", mape)
print("R²   :", r2)

MSE  : 31308.66995957857
RMSE : 176.94256118746154
MAE  : 127.8516195648178
MAPE : 0.25761173413837657
R²   : 0.9997223768098071


In [46]:
from src.database.postgres import get_connection

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE UNIQUE INDEX IF NOT EXISTS
                idx_model_metrics_unique

            ON energy.model_metrics (
                model_name,
                evaluation_period
            );
        """)

        cur.execute(
            """
            INSERT INTO energy.model_metrics (
                model_name,
                evaluation_period,
                mse,
                rmse,
                mae,
                mape,
                r2
            )

            VALUES (
                %s, %s, %s, %s, %s, %s, %s
            )

            ON CONFLICT (
                model_name,
                evaluation_period
            )

            DO UPDATE SET
                mse = EXCLUDED.mse,
                rmse = EXCLUDED.rmse,
                mae = EXCLUDED.mae,
                mape = EXCLUDED.mape,
                r2 = EXCLUDED.r2;
            """,

            (
                MODEL_NAME,
                "2025-development",
                mse,
                rmse,
                mae,
                mape,
                r2
            )
        )

    conn.commit()

print("Métriques enregistrées.")

Métriques enregistrées.


In [47]:
with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                model_name,
                evaluation_period,
                mse,
                rmse,
                mae,
                mape,
                r2

            FROM energy.model_metrics;
        """)

        rows = cur.fetchall()

metrics_sql = pd.DataFrame(
    rows,
    columns=[
        "model",
        "period",
        "MSE",
        "RMSE",
        "MAE",
        "MAPE",
        "R2"
    ]
)

display(metrics_sql)

,model,period,MSE,RMSE,MAE,MAPE,R2
0,Ridge + MLP1 + Patch Transformer + Refined Eve...,2025-development,31308.66996,176.942561,127.85162,0.257612,0.999722


In [48]:
pred_df = pd.DataFrame({

    "timestamp_utc":
        pd.to_datetime(
            block7_state["index_2025"],
            utc=True
        ),

    "prediction_mw":
        np.asarray(
            block7_state["prediction_2025"]
        ),

    "actual_mw":
        np.asarray(
            block7_state["y_2025"]
        )
})

pred_df["model_name"] = MODEL_NAME
pred_df["data_split"] = "development"

print(
    "Nombre de prédictions :",
    len(pred_df)
)

print(
    "Période :",
    pred_df["timestamp_utc"].min(),
    "→",
    pred_df["timestamp_utc"].max()
)

print(
    "Valeurs manquantes :",
    pred_df[
        ["prediction_mw", "actual_mw"]
    ].isna().sum().sum()
)

assert len(pred_df) == 17517

assert (
    pred_df["timestamp_utc"].max()
    <
    pd.Timestamp(
        "2026-01-01",
        tz="UTC"
    )
)

print("Contrôles validés.")

Nombre de prédictions : 17517
Période : 2025-01-01 00:00:00+00:00 → 2025-12-31 22:00:00+00:00
Valeurs manquantes : 0
Contrôles validés.


In [49]:
import time

start = time.perf_counter()

with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SET statement_timeout = '5min';

            CREATE UNIQUE INDEX IF NOT EXISTS
                idx_predictions_unique

            ON energy.predictions (
                timestamp_utc,
                model_name,
                data_split
            );
        """)

        cur.execute("""
            CREATE TEMP TABLE staging_predictions (
                timestamp_utc TIMESTAMPTZ,
                model_name VARCHAR(150),
                prediction_mw DOUBLE PRECISION,
                actual_mw DOUBLE PRECISION,
                data_split VARCHAR(30)
            );
        """)

        with cur.copy("""
            COPY staging_predictions (
                timestamp_utc,
                model_name,
                prediction_mw,
                actual_mw,
                data_split
            )
            FROM STDIN
        """) as copy:

            for row in pred_df[
                [
                    "timestamp_utc",
                    "model_name",
                    "prediction_mw",
                    "actual_mw",
                    "data_split"
                ]
            ].itertuples(
                index=False,
                name=None
            ):

                copy.write_row(row)

        cur.execute("""
            INSERT INTO energy.predictions (
                timestamp_utc,
                model_name,
                prediction_mw,
                actual_mw,
                data_split
            )

            SELECT
                timestamp_utc,
                model_name,
                prediction_mw,
                actual_mw,
                data_split

            FROM staging_predictions

            ON CONFLICT (
                timestamp_utc,
                model_name,
                data_split
            )

            DO UPDATE SET
                prediction_mw =
                    EXCLUDED.prediction_mw,

                actual_mw =
                    EXCLUDED.actual_mw;
        """)

    conn.commit()

import_time = (
    time.perf_counter()
    - start
)

print(
    f"Import terminé en {import_time:.3f} s."
)

Import terminé en 0.888 s.


In [50]:
with get_connection() as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                COUNT(*),
                MIN(timestamp_utc),
                MAX(timestamp_utc),

                AVG(
                    ABS(
                        actual_mw
                        - prediction_mw
                    )
                ) AS mae,

                SQRT(
                    AVG(
                        POWER(
                            actual_mw
                            - prediction_mw,
                            2
                        )
                    )
                ) AS rmse

            FROM energy.predictions

            WHERE
                model_name = %s

            AND
                data_split = 'development';
        """,
        (MODEL_NAME,))

        result = cur.fetchone()


print(
    "Nombre de prédictions SQL :",
    result[0]
)

print(
    "Début :",
    result[1]
)

print(
    "Fin :",
    result[2]
)

print(
    "MAE SQL :",
    result[3]
)

print(
    "RMSE SQL :",
    result[4]
)

Nombre de prédictions SQL : 17517
Début : 2025-01-01 00:00:00+00:00
Fin : 2025-12-31 22:00:00+00:00
MAE SQL : 127.85161956481734
RMSE SQL : 176.94256118746134


| Élément                               |                     Résultat |
| ------------------------------------- | ---------------------------: |
| Observations consommation             |                  **227 567** |
| Observations météo                    |                  **736 416** |
| Séries météo                          |                        **6** |
| Dataset ML SQL final                  |           **227 231 lignes** |
| Variables du dataset SQL              |                       **23** |
| NULL critiques                        |                        **0** |
| Requête janvier 2025                  |                 **0.677 ms** |
| Plan période courte                   |               **Index Scan** |
| Agrégation table complète             | **Parallel Sequential Scan** |
| Agrégation complète                   |               **136.696 ms** |
| Extraction SQL 2025 médiane           |                 **0.0813 s** |
| Extraction pandas RAM médiane         |                 **0.0590 s** |
| Prédictions stockées                  |                   **17 517** |
| MSE — Mean Squared Error              |               **31 308.670** |
| RMSE — Root Mean Squared Error        |               **176.943 MW** |
| MAE — Mean Absolute Error             |               **127.852 MW** |
| MAPE — Mean Absolute Percentage Error |                 **0.2576 %** |
| \(R^2\)                               |                **0.9997224** |
| MAE recalculée en SQL                 |            **127.851620 MW** |
| RMSE recalculée en SQL                |            **176.942561 MW** |
| Données 2026 utilisées                |                      **Non** |


## Conclusion finale — Bloc 8 : SQL & Energy Data Engineering

Le Bloc 8 transforme le projet de prévision énergétique en une
architecture de Data Engineering basée sur PostgreSQL.

La base contient désormais quatre familles d'informations :

- les données de consommation ;
- les données météorologiques ;
- les prédictions des modèles ;
- les métriques d'évaluation.

La série de consommation contient :

$$
227\,567
$$

observations entre janvier 2013 et décembre 2025.

Les données météorologiques contiennent :

$$
736\,416
$$

observations réparties entre cinq villes françaises et une température
moyenne nationale.

Les timestamps sont stockés en UTC tandis que les variables calendaires
sont calculées en heure locale française.

Les principales variables temporelles sont générées directement dans
PostgreSQL grâce aux fonctions fenêtre :

$$
LAG,\qquad AVG,\qquad STDDEV.
$$

Pour une fréquence de 30 minutes, les retards utilisés incluent :

$$
y_{t-1},\quad y_{t-2},\quad y_{t-48},\quad y_{t-336},
$$

correspondant respectivement à 30 minutes, 1 heure, 24 heures et
7 jours.

Les statistiques mobiles sont calculées uniquement sur les valeurs
passées afin d'éviter toute fuite de données.

La météo horaire est alignée avec la consommation à 30 minutes par une
jointure temporelle utilisant la dernière observation disponible.

Le résultat du Feature Engineering est matérialisé dans :

`energy.ml_features`

puis exposé à travers une vue destinée au Machine Learning :

`energy.ml_dataset`.

Cette vue contient :

$$
227\,231
$$

observations et aucune valeur manquante sur les variables critiques.

L'étude des plans d'exécution montre également que PostgreSQL choisit
correctement sa stratégie selon la requête.

Une sélection restreinte à janvier 2025 utilise un `Index Scan` avec
un temps d'exécution d'environ :

$$
0.677\text{ ms}.
$$

Une agrégation nécessitant la lecture de toute la table utilise au
contraire un `Parallel Sequential Scan`.

La couche PostgreSQL est reliée à Python par un module réutilisable :

`src/database/postgres.py`

permettant aux futurs composants du projet de récupérer directement
les données nécessaires.

Enfin, les performances et les prédictions du modèle retenu au Bloc 7 :

`Ridge + MLP1 + Patch Transformer + Refined Event Expert`

ont été persistées dans PostgreSQL.

Sur la période de développement 2025, le modèle obtient :

$$
MSE = 31\,308.67
$$

$$
RMSE = 176.943\text{ MW}
$$

$$
MAE = 127.852\text{ MW}
$$

$$
MAPE = 0.2576\%
$$

$$
R^2 = 0.9997224.
$$

Les métriques ont ensuite été recalculées indépendamment en SQL à partir
des 17 517 prédictions persistées.

PostgreSQL retrouve :

$$
MAE_{SQL}=127.851620\text{ MW}
$$

et

$$
RMSE_{SQL}=176.942561\text{ MW},
$$

ce qui correspond aux valeurs Python à l'arrondi numérique près.

La chaîne complète devient donc :

$$
\boxed{
Raw\ Data
\rightarrow PostgreSQL
\rightarrow Feature\ Engineering
\rightarrow Machine\ Learning
\rightarrow Predictions
\rightarrow PostgreSQL
}
$$

L'année 2026 reste totalement exclue du processus de développement et
demeure réservée au véritable test final.